# Testing Nodes with `run_to_targets`

`run_to_targets()` executes only the upstream nodes needed to deliver data to your target node, then returns the input salvos that would fire at the target — **without executing it**. This lets you:

1. Inspect exactly what a node would receive as input
2. Test the node function directly with real upstream data
3. Debug pipeline issues by examining intermediate values

**Pipeline:** `fetch_data → validate → transform → enrich → format_output`

## Setup

In [1]:
from pathlib import Path
from netrun.core import Net, NetConfig

config = NetConfig.from_file(Path("main.netrun.json"))

print("Pipeline nodes:")
for node in config.graph.nodes:
    print(f"  {node.name}")
print("\nEdges:")
for edge in config.graph.edges:
    print(f"  {edge.source_str} → {edge.target_str}")

Pipeline nodes:
  fetch_data
  validate
  transform
  enrich
  format_output

Edges:
  fetch_data.out → validate.data
  validate.out → transform.data
  transform.out → enrich.data
  enrich.out → format_output.data


## 1. Full Pipeline Run (baseline)

First, run the full pipeline to see the end-to-end result.

In [ ]:
async with Net(config) as net:
    net.inject_data("fetch_data", "url", ["https://example.com/data"])
    await net.run_until_blocked()

    results = net.flush_output_queue("results")
    print(results[0])
    print()
    net.logs.print_all(include_timestamps=False)

## 2. Inspect Input to `transform`

Now suppose we want to debug or test the `transform` node. Instead of running the whole pipeline, we use `run_to_targets("transform")` to:
- Execute only `fetch_data` and `validate` (the upstream nodes)
- Capture the input salvo at `transform` without executing it

In [ ]:
async with Net(config) as net:
    net.inject_data("fetch_data", "url", ["https://example.com/data"])
    salvos = await net.run_to_targets("transform")

    salvo = salvos[0]
    print(f"Target: {salvo.node_name} (condition: {salvo.salvo_condition})")
    print(f"Epoch ID: {salvo.epoch_id}")
    print()

    # Inspect the actual data that transform would receive
    input_data = salvo.packets["data"][0]
    print(f"Source: {input_data['source']}")
    print(f"Records ({len(input_data['records'])}):")
    for r in input_data["records"]:
        print(f"  {r}")
    print()
    print("Only upstream nodes ran:")
    net.logs.print_all(include_timestamps=False)

## 3. Test a Node Function Directly

With the captured input, we can call the node function directly — useful for unit testing or debugging.

In [4]:
from nodes import transform as transform_func

async with Net(config) as net:
    net.inject_data("fetch_data", "url", ["https://example.com/data"])
    salvos = await net.run_to_targets("transform")

    # Extract the input and call the function directly
    input_data = salvos[0].packets["data"][0]
    result = transform_func(input_data, print=print)

    print(f"\nReturned {len(result['records'])} records:")
    for r in result["records"]:
        print(f"  {r['name']}: score={r['score']}, normalized={r['score_normalized']}, grade={r['grade']}")

Transforming 1 records
Transformation complete

Returned 1 records:
  Alice: score=85, normalized=0.85, grade=B


## 4. Inspect Input to the Final Node

Target `format_output` (the last node). All upstream nodes execute, but `format_output` itself does not.

In [5]:
async with Net(config) as net:
    net.inject_data("fetch_data", "url", ["https://example.com/data"])
    salvos = await net.run_to_targets("format_output")

    data = salvos[0].packets["data"][0]
    print(f"format_output would receive {len(data['records'])} enriched records:")
    for r in data["records"]:
        print(f"  {r}")

format_output would receive 1 enriched records:
  {'name': 'Alice', 'age': 30, 'score': 85, 'score_normalized': 0.85, 'grade': 'B', 'label': 'Alice (Grade B)'}


## 5. Target a Source Node

`fetch_data` has no upstream nodes. Targeting it returns the injected salvo immediately — nothing executes.

In [6]:
async with Net(config) as net:
    net.inject_data("fetch_data", "url", ["https://example.com/data"])
    salvos = await net.run_to_targets("fetch_data")

    print(f"Target: {salvos[0].node_name}")
    print(f"Packets: {salvos[0].packets}")
    print(f"Upstream epochs executed: {len(net.epochs)}")

Target: fetch_data
Packets: {'url': ['https://example.com/data']}
Upstream epochs executed: 0
